In [1]:
import pandas as pd
import json
import re
import numpy as np

# -----------------------------
# Load JSON File
# -----------------------------
with open("global_freelancers_nosql.json", "r", encoding="utf-8") as f:
    data = json.load(f)

# Flatten nested JSON
records = []
for item in data:
    personal = item.get("personal_info", {})
    prof = item.get("professional_info", {})
    records.append({
        "freelancer_id": item.get("freelancer_id"),
        "name": personal.get("name"),
        "gender": personal.get("gender"),
        "age": personal.get("age"),
        "country": personal.get("country"),
        "language": personal.get("language"),
        "primary_skill": prof.get("primary_skill"),
        "years_of_experience": prof.get("years_of_experience"),
        "hourly_rate_usd": prof.get("hourly_rate_usd"),
        "rating": prof.get("rating"),
        "is_active": prof.get("is_active"),
        "client_satisfaction": prof.get("client_satisfaction")
    })

df = pd.DataFrame(records)

# -----------------------------
# 1️⃣ Sort freelancer_id ascending
# -----------------------------
df = df.sort_values(by="freelancer_id", ascending=True)

# -----------------------------
# 2️⃣ Remove null / NaN / blank in name
# -----------------------------
df["name"] = df["name"].replace(["", " ", "NaN", "nan", "NAN"], np.nan)
df = df.dropna(subset=["name"])

# -----------------------------
# 3️⃣ Clean gender
# -----------------------------
def clean_gender(x):
    if pd.isna(x): return np.nan
    x = str(x).strip().lower()
    if x in ["m", "male"]: return "Male"
    elif x in ["f", "female"]: return "Female"
    else: return np.nan

df["gender"] = df["gender"].apply(clean_gender)
df = df.dropna(subset=["gender"])

# -----------------------------
# 4️⃣ Clean age (remove blanks)
# -----------------------------
df["age"] = pd.to_numeric(df["age"], errors="coerce")
df = df.dropna(subset=["age"])

# -----------------------------
# 5️⃣ Remove nulls in country, language, primary_skill
# -----------------------------
for col in ["country", "language", "primary_skill"]:
    df[col] = df[col].replace(["", " ", "NaN", "nan", "NAN"], np.nan)
    df = df.dropna(subset=[col])

# -----------------------------
# 6️⃣ years_of_experience - null/blank => 0
# -----------------------------
df["years_of_experience"] = pd.to_numeric(df["years_of_experience"], errors="coerce").fillna(0)

# -----------------------------
# 7️⃣ Clean hourly_rate_usd
# -----------------------------
def clean_rate(x):
    if pd.isna(x): return 0
    x = str(x)
    x = re.sub(r"[^0-9.]", "", x)  # Remove $, USD, etc.
    try:
        return float(x)
    except:
        return 0

df["hourly_rate_usd"] = df["hourly_rate_usd"].apply(clean_rate)

# -----------------------------
# 8️⃣ rating - null/blank => 0
# -----------------------------
df["rating"] = pd.to_numeric(df["rating"], errors="coerce").fillna(0)

# -----------------------------
# 9️⃣ is_active - normalize to yes/no
# -----------------------------
def clean_active(x):
    if pd.isna(x): return "no"
    x = str(x).strip().lower()
    if x in ["1", "yes", "y", "true"]: return "yes"
    elif x in ["0", "no", "n", "false"]: return "no"
    else: return "no"

df["is_active"] = df["is_active"].apply(clean_active)

# -----------------------------
# 🔟 client_satisfaction - null/blank => 0%
# -----------------------------
def clean_satisfaction(x):
    if pd.isna(x): return "0%"
    x = str(x).strip()
    if x in ["", "NaN", "nan", "NAN", "NA"]: return "0%"
    if not x.endswith("%"):
        x = re.sub(r"[^0-9]", "", x)
        x = x + "%"
    return x

df["client_satisfaction"] = df["client_satisfaction"].apply(clean_satisfaction)

# -----------------------------
# ✅ Save final cleaned dataset
# -----------------------------
df.to_csv("global_freelancers_clean.csv", index=False)
print("✅ Cleaned dataset saved as 'global_freelancers_clean.csv'")

# Show preview
print("\nPreview of cleaned data:")
print(df.head(10))


✅ Cleaned dataset saved as 'global_freelancers_clean.csv'

Preview of cleaned data:
  freelancer_id             name  gender   age        country    language  \
0      FL250001  Ms. Nicole Kidd  Female  52.0          Italy     Italian   
1      FL250002   Vanessa Garcia  Female  52.0      Australia     English   
2      FL250003      Juan Nelson    Male  53.0        Germany      German   
3      FL250004   Amanda Spencer  Female  38.0      Australia     English   
4      FL250005  Lynn Curtis DDS  Female  53.0        Germany      German   
5      FL250006     Lisa Johnson  Female  59.0    Netherlands       Dutch   
6      FL250007       Eric Myers    Male  52.0      Indonesia  Indonesian   
7      FL250008     Ricky Graham    Male  43.0          Italy     Italian   
8      FL250009      Sean Martin    Male  26.0  United States     English   
9      FL250010    Matthew Lloyd    Male  52.0         Turkey     Turkish   

            primary_skill  years_of_experience  hourly_rate_usd  rat